In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install rank-bm25

#Konfig dan Logging

In [5]:
import os, sys, json, math, logging
import numpy as np
import pandas as pd
from collections import defaultdict
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm
import re

# Logging profesional
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

# Konfigurasi terpusat
CONFIG = {
    "base_dir": "/content/drive/MyDrive/Semester6/TKI/BooksSearch",
    "min_doc_length": 100,
    "top_k_eval": 10,
    "bm25_k1": 1.2,
    "bm25_b": 0.75,
    "remove_stopwords": True
}

# Paths
RAW_PATH = os.path.join(CONFIG["base_dir"], "data", "raw", "book_data.csv")
PROC_PATH = os.path.join(CONFIG["base_dir"], "data", "processed", "cleaned_corpus.csv")
GT_PATH = os.path.join(CONFIG["base_dir"], "data", "ground_truth.json")

# Validasi path kritis
for p in [RAW_PATH, GT_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"🚨 File wajib tidak ditemukan: {p}")
os.makedirs(os.path.dirname(PROC_PATH), exist_ok=True)

# NLTK
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
STOP_WORDS = set(stopwords.words('english'))

#Load Data dan Validasi

In [6]:
def load_and_validate_dataset(raw_path: str, min_words: int) -> pd.DataFrame:
    logger.info("📥 Memuat dataset mentah...")
    df = pd.read_csv(raw_path, encoding='latin1')

    required = ['book_title', 'book_desc', 'genres', 'book_authors', 'book_rating']
    missing = [c for c in required if c not in df.columns]
    if missing: raise ValueError(f"Kolom kritis hilang: {missing}")

    df = df[required].dropna(subset=['book_title', 'book_desc']).copy()
    logger.info(f"🔍 Memfilter dokumen < {min_words} kata...")
    df['word_count'] = df['book_desc'].astype(str).apply(lambda x: len(x.split()))
    df = df[df['word_count'] >= min_words].copy()

    if len(df) < 50:
        logger.warning("⚠️ Dataset sangat kecil setelah filter. Pastikan threshold sesuai.")
    logger.info(f"✅ {len(df)} dokumen lolos validasi.")
    return df

#Preprocessing Pipeline

In [7]:
def preprocess_text(text: str, cfg: dict) -> str:
    if pd.isna(text) or not str(text).strip(): return ""
    text = str(text).lower()                          # Case folding
    text = re.sub(r'<.*?>|http\S+|www\S+|[^a-z\s]', ' ', text)  # Cleaning
    text = re.sub(r'\s+', ' ', text).strip()          # Normalisasi spasi
    tokens = word_tokenize(text)                      # Tokenisasi NLTK
    if cfg.get("remove_stopwords", True):
        tokens = [t for t in tokens if t not in STOP_WORDS]  # Stopword removal
    return ' '.join(tokens)

def build_corpus(df: pd.DataFrame) -> pd.DataFrame:
    logger.info("🧹 Membangun corpus & preprocessing...")
    df['raw_corpus'] = (df['book_title'].fillna('') + ' ' +
                        df['book_authors'].fillna('') + ' ' +
                        df['genres'].fillna('') + ' ' +
                        df['book_desc'].fillna(''))

    tqdm.pandas(desc="Preprocessing")
    df['clean_corpus'] = df['raw_corpus'].progress_apply(lambda x: preprocess_text(x, CONFIG))

    # Hapus dokumen yang kosong setelah cleaning
    df = df[df['clean_corpus'].str.strip().astype(bool)].reset_index(drop=True)

    out = df[['book_title', 'book_authors', 'genres', 'book_rating', 'clean_corpus']]
    out.to_csv(PROC_PATH, index=False, encoding='utf-8')
    logger.info(f"💾 Corpus tersimpan: {PROC_PATH} ({len(df)} dokumen)")
    return df

In [8]:
# Validasi Ground Truth
import json
import pandas as pd

df_check = pd.read_csv(PROC_PATH)
with open(GT_PATH, 'r', encoding='utf-8') as f:
    gt_check = json.load(f)

available = set(df_check['book_title'].unique())
print("🔍 Validasi exact-match ground_truth.json:\n")
for q in gt_check['queries']:
    miss = [d for d in q['relevant_docs'] if d not in available]
    status = "✅" if not miss else f"❌ {len(miss)} judul missing"
    print(f"{q['id']}: {status}")

🔍 Validasi exact-match ground_truth.json:

Q1: ✅
Q2: ✅
Q3: ✅
Q4: ✅
Q5: ✅
Q6: ✅
Q7: ✅
Q8: ✅
Q9: ✅
Q10: ✅


#BM25 Model

In [9]:
class STKISearchEngine:
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.corpus_tokens = [doc.split() for doc in df['clean_corpus']]
        self.bm25 = BM25Okapi(self.corpus_tokens, k1=CONFIG["bm25_k1"], b=CONFIG["bm25_b"])
        logger.info("✅ Indeks BM25 dibangun (k1=1.2, b=0.75 | Probabilistic Model)")

    def search(self, query: str, top_k: int = 10) -> pd.DataFrame:
        if not query or not str(query).strip():
            return pd.DataFrame()

        q_tokens = preprocess_text(str(query), CONFIG).split()
        if not q_tokens:
            logger.warning("Query kosong setelah preprocessing.")
            return pd.DataFrame()

        scores = self.bm25.get_scores(q_tokens)
        top_idx = np.argsort(scores)[::-1][:top_k]

        results = []
        for rank, idx in enumerate(top_idx, 1):
            if scores[idx] <= 1e-9: break  # Threshold noise BM25
            results.append({
                'rank': rank,
                'book_title': self.df.iloc[idx]['book_title'],
                'authors': self.df.iloc[idx]['book_authors'],
                'genres': self.df.iloc[idx]['genres'],
                'rating': self.df.iloc[idx]['book_rating'],
                'relevance_score': round(float(scores[idx]), 4)
            })
        return pd.DataFrame(results)

#Evaluasi

In [10]:
def load_ground_truth(gt_path: str) -> dict:
    with open(gt_path, 'r', encoding='utf-8') as f:
        gt = json.load(f)
    if 'queries' not in gt or len(gt['queries']) < 10:
        raise ValueError("Ground truth wajib berisi ≥10 query dengan konteks berbeda.")
    return gt

def precision_at_k(retrieved: list, relevant: set, k: int) -> float:
    return sum(1 for d in retrieved[:k] if d in relevant) / k if k > 0 else 0.0

def average_precision(retrieved: list, relevant: set) -> float:
    if not relevant: return 0.0
    hits, sum_prec = 0.0, 0.0
    for i, d in enumerate(retrieved, 1):
        if d in relevant:
            hits += 1
            sum_prec += hits / i
    return sum_prec / len(relevant)

def ndcg_at_k(retrieved: list, relevant: set, k: int) -> float:
    if k == 0: return 0.0
    dcg = sum((1.0 if d in relevant else 0.0) / math.log2(i + 1) for i, d in enumerate(retrieved[:k], 1))
    ideal = [1.0]*min(len(relevant), k) + [0.0]*max(0, k-len(relevant))
    idcg = sum((2**r - 1) / math.log2(i + 1) for i, r in enumerate(ideal, 1))
    return dcg / idcg if idcg > 0 else 0.0

def normalize_title(title: str) -> str:
    """Normalisasi judul untuk pencocokan yang robust."""
    return re.sub(r'[^a-z0-9]', '', str(title).lower().strip())

def evaluate_system(engine: STKISearchEngine, gt: dict, top_k: int = 10) -> pd.DataFrame:
    logger.info("📊 Evaluasi sistem vs ground truth (10 query)...")
    metrics = defaultdict(list)

    for q in gt['queries']:
        # Normalisasi relevant_docs sekali per query
        rel_normalized = {normalize_title(d) for d in q['relevant_docs']}

        res = engine.search(q['text'], top_k=top_k)
        ret = res['book_title'].tolist() if not res.empty else []
        ret_normalized = [normalize_title(t) for t in ret]

        metrics['query_id'].append(q['id'])
        metrics['P@3'].append(precision_at_k(ret_normalized, rel_normalized, 3))
        metrics['P@5'].append(precision_at_k(ret_normalized, rel_normalized, 5))
        metrics['AP'].append(average_precision(ret_normalized, rel_normalized))
        metrics['NDCG@5'].append(ndcg_at_k(ret_normalized, rel_normalized, 5))

    df_met = pd.DataFrame(metrics)
    logger.info("✅ Evaluasi selesai.")
    return df_met

#Demo

In [12]:
if __name__ == "__main__":
    logger.info("🚀 Memulai Pipeline STKI Lengkap...")
    df = load_and_validate_dataset(RAW_PATH, CONFIG["min_doc_length"])
    df = build_corpus(df)
    engine = STKISearchEngine(df)

    gt = load_ground_truth(GT_PATH)
    eval_df = evaluate_system(engine, gt, top_k=CONFIG["top_k_eval"])

    print("\n📈 HASIL EVALUASI SISTEM:")
    print(f"MAP (Mean Average Precision): {eval_df['AP'].mean():.4f}")
    print(f"Avg Precision@3: {eval_df['P@3'].mean():.4f}")
    print(f"Avg Precision@5: {eval_df['P@5'].mean():.4f}")
    print(f"Avg NDCG@5: {eval_df['NDCG@5'].mean():.4f}")
    display(eval_df)

    print("\n🔎 Mode Pencarian Interaktif (ketik 'exit' untuk keluar)")
    while True:
        q = input("\nQuery: ").strip()
        if q.lower() in ['exit', 'keluar', '']: break
        res = engine.search(q, top_k=5)
        display(res) if not res.empty else print("Tidak ada hasil relevan.")

Preprocessing:   0%|          | 0/35894 [00:00<?, ?it/s]


📈 HASIL EVALUASI SISTEM:
MAP (Mean Average Precision): 0.3081
Avg Precision@3: 0.2667
Avg Precision@5: 0.2200
Avg NDCG@5: 0.3331


,query_id,P@3,P@5,AP,NDCG@5
0,Q1,0.000000,0.2,0.273016,0.181542
1,Q2,0.333333,0.2,0.166667,0.296082
2,Q3,0.000000,0.0,0.122222,0.000000
3,Q4,0.000000,0.0,0.000000,0.000000
4,Q5,0.000000,0.0,0.000000,0.000000
5,Q6,0.666667,0.6,0.866667,0.946902
6,Q7,0.333333,0.4,0.625000,0.671386
7,Q8,0.666667,0.4,0.388889,0.530721
8,Q9,0.333333,0.2,0.305556,0.234639
9,Q10,0.333333,0.2,0.333333,0.469279



🔎 Mode Pencarian Interaktif (ketik 'exit' untuk keluar)

Query: sophie


,rank,book_title,authors,genres,rating,relevance_score
0,1,Embrace,Annalise Grey,Shapeshifters|Werewolves|Fantasy|Paranormal|Ro...,4.25,9.7736
1,2,Spell Bound,Rachel Hawkins,Young Adult|Fantasy|Fantasy|Paranormal|Fantasy...,4.08,9.7309
2,3,Sophie & Carter,Chelsea Fine,Young Adult|Romance|Contemporary|Young Adult|H...,3.83,9.6732
3,4,Sophie's Misfortunes,Comtesse de SÃ©gur|Stephanie Smee|Simon Sturge,Classics|Cultural|France|Childrens|Fiction,3.90,9.6050
4,5,Snapped,Laura Griffin,Romance|Romantic Suspense|Romance|Suspense|Mys...,4.17,9.5843



Query: herry
Tidak ada hasil relevan.

Query: harry


,rank,book_title,authors,genres,rating,relevance_score
0,1,Harry Potter and the Deathly Hallows,J.K. Rowling,Fantasy|Young Adult|Fiction,4.62,9.4111
1,2,Harry Potter and the Deathly Hallows,J.K. Rowling|Mary GrandPrÃ©,Fantasy|Young Adult|Fiction,4.62,9.2011
2,3,Mugglenet.Com's What Will Happen in Harry Pott...,Ben Schoen|Andy Gordon|Gretchen Stull|Emerson ...,Nonfiction,4.20,9.1849
3,4,Harry Potter and the Deathly Hallows,J.K. Rowling,Fantasy|Young Adult|Fiction,4.62,9.0637
4,5,Harry Potter and the Chamber of Secrets,J.K. Rowling|Mary GrandPrÃ©,Fantasy|Young Adult|Fiction,4.40,9.0551



Query: 
